In [1]:
import os
import sys
from pathlib import Path

sys.path.append(str(Path(os.getcwd()).resolve()))

import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
from tqdm import tqdm
from torchsummary import summary
from PIL import Image

import torch
import torch.nn.functional as F                          # Functions like ReLU
import torch.optim as optim                              # Optimizers like Adam
import torch.nn as nn                                    # Neural Network Modules


from load_data import CatDogDataLoadandSave, CatandDogDataLoader
from vgg16_model import VGG16

### Load data and Save dataset as ubyte format


In [ ]:


data_path = r'/Users/gimoon/Documents/GitHub/Data'
train_data_path = os.path.join(data_path, "cat-and-dog/training_set/")
test_data_path  = os.path.join(data_path, "cat-and-dog/test_set/")

# load data 
ds_train = CatDogDataLoadandSave(data_dir=train_data_path)
ds_test = CatDogDataLoadandSave(data_dir=test_data_path)

# save them as ubyte format 
output_directory = "/Users/gimoon/Documents/GitHub/Data/cat-and-dog/ubyte_format"
ds_train.save_as_ubyte(output_dir=output_directory, img_size=(224, 224), prefix="catdog_train")
ds_test.save_as_ubyte(output_dir=output_directory, img_size=(224, 224), prefix="catdog_test")


### create torch data loader 

In [3]:
output_directory = "/Users/gimoon/Documents/GitHub/Data/cat-and-dog/ubyte_format"

ds_train = CatandDogDataLoader(raw_folder=output_directory, train=True)
train_loader = torch.utils.data.DataLoader(ds_train, batch_size=64, shuffle=True)  

Loaded 8005 images of shape 224x224x3
Loaded 2049 labels


### Build VGG16 architecture 

In [4]:
device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
print(device)

model = VGG16(3, 2).to(device)

mps


In [5]:
learning_rate = 1e-4
num_epoches = 20 

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

In [6]:
for epoch in range(num_epoches):
    ProgressBar = tqdm(enumerate(train_loader), total=len(train_loader))

    model.train()
    running_loss = 0.0
    for batch_idx, (inputs, labels) in ProgressBar:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()


        #Update Progress bar
        ProgressBar.set_description(f'Epoch [{epoch+1}]')
        ProgressBar.set_postfix(loss=loss.item())

Epoch [2]:  10%|█         | 13/126 [00:56<08:15,  4.38s/it, loss=0.691]


KeyboardInterrupt: 